###  Setup and Load Data:

In [1]:
# Cell 1: Setup and Load Data
import pandas as pd
import numpy as np
import random
from collections import defaultdict
import json
import sys

# Load your similarity results
file_path = "results/similarity_analysis_150000_results.csv"
print(f"📁 Loading data from: {file_path}")
df = pd.read_csv(file_path)

print(f"✅ Loaded {len(df):,} similar pairs")
print(f"\n📊 Data preview:")
print(df.head())
print(f"\n📈 Data info:")
print(df.info())

# Define similarity thresholds
POSITIVE_JACCARD_MIN = 0.9
POSITIVE_COSINE_MIN = 0.7
NEGATIVE_JACCARD_MAX = 0.1
NEGATIVE_COSINE_MAX = 0.3

print(f"\n🎯 THRESHOLDS:")
print(f"   Positive: Jaccard ≥ {POSITIVE_JACCARD_MIN}, Cosine ≥ {POSITIVE_COSINE_MIN}")
print(f"   Negative: Jaccard ≤ {NEGATIVE_JACCARD_MAX}, Cosine ≤ {NEGATIVE_COSINE_MAX}")

📁 Loading data from: results/similarity_analysis_150000_results.csv
✅ Loaded 97,314 similar pairs

📊 Data preview:
                                          base_query  \
0  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...   
1  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...   
2  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...   
3  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...   
4  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...   

                                       similar_query  jaccard_score  \
0  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...            1.0   
1  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...            1.0   
2  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...            1.0   
3  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...            1.0   
4  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...            1.0   

   cosine_score  combined_score  base_query_length  similar_query_length  
0           1.0             1.0               

### Analyze Current Similarity Pairs:

In [2]:
# Cell 2: Analyze Current Similarity Pairs
print("🔍 ANALYZING CURRENT SIMILARITY PAIRS")
print("="*60)

# 1. Count unique queries
all_base_queries = set(df['base_query'].unique())
all_similar_queries = set(df['similar_query'].unique())
all_queries = all_base_queries.union(all_similar_queries)

print(f"📊 Unique query statistics:")
print(f"   Unique base queries: {len(all_base_queries):,}")
print(f"   Unique similar queries: {len(all_similar_queries):,}")
print(f"   Total unique queries: {len(all_queries):,}")

# 2. Check which pairs meet positive criteria
df['is_positive'] = (df['jaccard_score'] >= POSITIVE_JACCARD_MIN) & (df['cosine_score'] >= POSITIVE_COSINE_MIN)
positive_pairs = df[df['is_positive']]
negative_candidate_pairs = df[~df['is_positive']]

print(f"\n✅ Positive pairs (meets both thresholds): {len(positive_pairs):,}")
print(f"   Percentage of total: {len(positive_pairs)/len(df)*100:.1f}%")

# 3. Check if any pairs are already negative (meet negative criteria)
df['is_negative'] = (df['jaccard_score'] <= NEGATIVE_JACCARD_MAX) & (df['cosine_score'] <= NEGATIVE_COSINE_MAX)
negative_pairs_existing = df[df['is_negative']]

print(f"\n❌ Existing negative pairs: {len(negative_pairs_existing):,}")
print(f"   Percentage of total: {len(negative_pairs_existing)/len(df)*100:.1f}%")

# 4. Show distribution of base queries with positive pairs
base_query_positive_counts = positive_pairs['base_query'].value_counts()
print(f"\n📋 Base queries with positive pairs:")
print(f"   Total base queries with ≥1 positive: {len(base_query_positive_counts):,}")
print(f"   Average positive per base: {base_query_positive_counts.mean():.1f}")
print(f"   Max positive for a base: {base_query_positive_counts.max()}")
print(f"   Min positive for a base: {base_query_positive_counts.min()}")

# Show sample
if len(base_query_positive_counts) > 0:
    print(f"\n🔍 Sample base query with positive pairs:")
    sample_base = base_query_positive_counts.index[0]
    sample_positives = positive_pairs[positive_pairs['base_query'] == sample_base]
    print(f"   Base query: '{sample_base[:100]}...'")
    print(f"   Has {len(sample_positives)} positive pairs")
    for i, row in sample_positives.head(3).iterrows():
        print(f"     Positive {i+1}: J={row['jaccard_score']:.3f}, C={row['cosine_score']:.3f}")
        print(f"     Query: '{row['similar_query'][:80]}...'")

🔍 ANALYZING CURRENT SIMILARITY PAIRS
📊 Unique query statistics:
   Unique base queries: 11,791
   Unique similar queries: 11,015
   Total unique queries: 15,190

✅ Positive pairs (meets both thresholds): 97,187
   Percentage of total: 99.9%

❌ Existing negative pairs: 0
   Percentage of total: 0.0%

📋 Base queries with positive pairs:
   Total base queries with ≥1 positive: 11,772
   Average positive per base: 8.3
   Max positive for a base: 20
   Min positive for a base: 1

🔍 Sample base query with positive pairs:
   Base query: 'PREFIX  dbpo: <http://dbpedia.org/ontology/>
PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-...'
   Has 20 positive pairs
     Positive 23: J=1.000, C=1.000
     Query: 'PREFIX  dbpo: <http://dbpedia.org/ontology/>
PREFIX  rdf:  <http://www.w3.org/19...'
     Positive 25: J=1.000, C=1.000
     Query: 'PREFIX  dbpo: <http://dbpedia.org/ontology/>
PREFIX  rdf:  <http://www.w3.org/19...'
     Positive 26: J=1.000, C=1.000
     Query: 'PREFIX  dbpo: <http

### Build Query Index and Similarity Maps:

In [3]:
# Cell 3: Build Query Index and Similarity Maps
print("🔨 BUILDING QUERY INDEX AND SIMILARITY MAPS")
print("="*60)

# Create dictionaries for quick lookup
base_to_similars = defaultdict(list)  # base -> list of (similar_query, jaccard, cosine)
query_to_index = {}  # query -> index
index_to_query = {}  # index -> query

# Assign indices to all unique queries
for idx, query in enumerate(all_queries):
    query_to_index[query] = idx
    index_to_query[idx] = query

print(f"✅ Indexed {len(query_to_index):,} unique queries")

# Build similarity matrix (sparse representation)
similarity_dict = defaultdict(dict)  # query_idx1 -> {query_idx2: (jaccard, cosine)}

# Fill with existing similarities from CSV
for _, row in df.iterrows():
    base_idx = query_to_index[row['base_query']]
    similar_idx = query_to_index[row['similar_query']]
    
    # Store both directions
    similarity_dict[base_idx][similar_idx] = (row['jaccard_score'], row['cosine_score'])
    similarity_dict[similar_idx][base_idx] = (row['jaccard_score'], row['cosine_score'])
    
    # Also store in base_to_similars for easy access
    base_to_similars[base_idx].append({
        'query_idx': similar_idx,
        'query': row['similar_query'],
        'jaccard': row['jaccard_score'],
        'cosine': row['cosine_score'],
        'is_positive': row['is_positive']
    })

print(f"✅ Built similarity dictionary with {len(similarity_dict):,} query connections")

# Also build a list of all query indices for random sampling
all_query_indices = list(query_to_index.values())
print(f"✅ Prepared {len(all_query_indices):,} query indices for sampling")

🔨 BUILDING QUERY INDEX AND SIMILARITY MAPS
✅ Indexed 15,190 unique queries
✅ Built similarity dictionary with 15,190 query connections
✅ Prepared 15,190 query indices for sampling


### Generate Negative Pairs:

In [4]:
# Cell 4: Generate Negative Pairs
print("🔄 GENERATING NEGATIVE PAIRS")
print("="*60)

# We need to create negative pairs that:
# 1. Are not already in our similarity data (or have low similarity)
# 2. Have Jaccard ≤ 0.1 AND Cosine ≤ 0.3
# 3. Don't duplicate existing positive pairs

negative_pairs = []
max_negatives_per_base = 20  # Maximum negative pairs per base query
total_negative_target = len(positive_pairs)  # Match number of positive pairs

print(f"Target: Generate ~{total_negative_target:,} negative pairs")

# We'll generate negatives for base queries that have positives
base_indices_with_positives = []
for base_idx, similars in base_to_similars.items():
    if any(s['is_positive'] for s in similars):
        base_indices_with_positives.append(base_idx)

print(f"Base queries with positives: {len(base_indices_with_positives):,}")

# Function to check if a pair is truly negative
def is_truly_negative(query1_idx, query2_idx):
    """Check if two queries are truly dissimilar"""
    # Check if we have existing similarity data
    if query2_idx in similarity_dict.get(query1_idx, {}):
        jaccard, cosine = similarity_dict[query1_idx][query2_idx]
        return jaccard <= NEGATIVE_JACCARD_MAX and cosine <= NEGATIVE_COSINE_MAX
    else:
        # We don't have data - assume negative (conservative)
        # In practice, you might want to compute or have pre-computed these
        return True

# Generate negative pairs
generated_count = 0
attempts = 0
max_attempts = len(base_indices_with_positives) * 1000

for base_idx in base_indices_with_positives:
    base_query = index_to_query[base_idx]
    
    # Get positive similars for this base (to avoid)
    positive_similar_indices = [s['query_idx'] for s in base_to_similars[base_idx] if s['is_positive']]
    
    # Try to find negative pairs
    negatives_found = 0
    for _ in range(100):  # Try up to 100 candidates per base
        if negatives_found >= max_negatives_per_base:
            break
            
        # Pick a random query that's not the base itself and not a positive
        candidate_idx = random.choice(all_query_indices)
        
        # Skip if: same query, is a positive similar, or we've already checked
        if (candidate_idx == base_idx or 
            candidate_idx in positive_similar_indices):
            continue
        
        # Check if this would be a good negative pair
        if is_truly_negative(base_idx, candidate_idx):
            negative_pairs.append({
                'base_query': base_query,
                'negative_query': index_to_query[candidate_idx],
                'base_idx': base_idx,
                'negative_idx': candidate_idx,
                'assumed_jaccard': 0.05,  # Conservative estimate
                'assumed_cosine': 0.15    # Conservative estimate
            })
            negatives_found += 1
            generated_count += 1
    
    attempts += 1
    if attempts % 1000 == 0:
        print(f"   Processed {attempts:,} bases, generated {generated_count:,} negatives")
    
    if generated_count >= total_negative_target:
        print(f"✅ Reached target of {total_negative_target:,} negative pairs")
        break

print(f"\n✅ Generated {len(negative_pairs):,} negative pairs")
print(f"   Average negatives per base: {len(negative_pairs)/len(base_indices_with_positives):.1f}")

# Create DataFrame for negative pairs
negative_df = pd.DataFrame(negative_pairs)
print(f"\n📋 Negative pairs sample:")
print(negative_df.head())

🔄 GENERATING NEGATIVE PAIRS
Target: Generate ~97,187 negative pairs
Base queries with positives: 11,772
   Processed 1,000 bases, generated 20,000 negatives
   Processed 2,000 bases, generated 40,000 negatives
   Processed 3,000 bases, generated 60,000 negatives
   Processed 4,000 bases, generated 80,000 negatives
✅ Reached target of 97,187 negative pairs

✅ Generated 97,200 negative pairs
   Average negatives per base: 8.3

📋 Negative pairs sample:
                                          base_query  \
0  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...   
1  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...   
2  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...   
3  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...   
4  PREFIX  dbpo: <http://dbpedia.org/ontology/>\n...   

                                      negative_query  base_idx  negative_idx  \
0  PREFIX  :     <http://dbpedia.org/resource/>\n...     14024          6457   
1  PREFIX  rdf:  <http://www.w3.org/1999/02/22-rd

### Create Final Dataset Structure:

In [5]:
# Cell 5: Create Final Dataset with Base, Positives, Negatives
print("📦 CREATING FINAL DATASET")
print("="*60)

# We'll create a dataset where each entry has:
# - base_query: string
# - positive_queries: list of strings (queries with J≥0.9, C≥0.7)
# - negative_queries: list of strings (queries with J≤0.1, C≤0.3)

final_dataset = []

# Group positive pairs by base query
positive_by_base = positive_pairs.groupby('base_query')

# Group negative pairs by base query
negative_by_base = negative_df.groupby('base_query')

# Get all base queries that have both positives and negatives
base_queries_with_both = set(positive_by_base.groups.keys()).intersection(
    set(negative_by_base.groups.keys())
)

print(f"Base queries with both positives and negatives: {len(base_queries_with_both):,}")

# Build final dataset entries
for base_query in base_queries_with_both:
    # Get positive queries for this base
    base_positives = positive_pairs[positive_pairs['base_query'] == base_query]
    positive_list = []
    for _, row in base_positives.iterrows():
        positive_list.append({
            'query': row['similar_query'],
            'jaccard_score': row['jaccard_score'],
            'cosine_score': row['cosine_score']
        })
    
    # Get negative queries for this base
    base_negatives = negative_df[negative_df['base_query'] == base_query]
    negative_list = []
    for _, row in base_negatives.iterrows():
        negative_list.append({
            'query': row['negative_query'],
            'assumed_jaccard': row['assumed_jaccard'],
            'assumed_cosine': row['assumed_cosine']
        })
    
    # Only include if we have at least some positives and negatives
    if len(positive_list) >= 1 and len(negative_list) >= 1:
        final_dataset.append({
            'base_query': base_query,
            'positive_queries': positive_list,
            'negative_queries': negative_list,
            'positive_count': len(positive_list),
            'negative_count': len(negative_list)
        })

print(f"\n✅ Created {len(final_dataset):,} dataset entries")
print(f"\n📊 Dataset statistics:")
if final_dataset:
    avg_positives = np.mean([entry['positive_count'] for entry in final_dataset])
    avg_negatives = np.mean([entry['negative_count'] for entry in final_dataset])
    print(f"   Average positives per entry: {avg_positives:.1f}")
    print(f"   Average negatives per entry: {avg_negatives:.1f}")
    print(f"   Min positives: {min([entry['positive_count'] for entry in final_dataset])}")
    print(f"   Max positives: {max([entry['positive_count'] for entry in final_dataset])}")
    print(f"   Min negatives: {min([entry['negative_count'] for entry in final_dataset])}")
    print(f"   Max negatives: {max([entry['negative_count'] for entry in final_dataset])}")

# Show example entry
if final_dataset:
    print(f"\n🔍 EXAMPLE ENTRY:")
    example = final_dataset[0]
    print(f"   Base query: '{example['base_query'][:100]}...'")
    print(f"   Positive queries: {example['positive_count']}")
    for i, pos in enumerate(example['positive_queries'][:2]):
        print(f"     Positive {i+1}: J={pos['jaccard_score']:.3f}, C={pos['cosine_score']:.3f}")
        print(f"     Query: '{pos['query'][:80]}...'")
    print(f"   Negative queries: {example['negative_count']}")
    for i, neg in enumerate(example['negative_queries'][:2]):
        print(f"     Negative {i+1}: J≈{neg['assumed_jaccard']:.3f}, C≈{neg['assumed_cosine']:.3f}")
        print(f"     Query: '{neg['query'][:80]}...'")

📦 CREATING FINAL DATASET
Base queries with both positives and negatives: 4,860

✅ Created 4,860 dataset entries

📊 Dataset statistics:
   Average positives per entry: 9.9
   Average negatives per entry: 20.0
   Min positives: 1
   Max positives: 20
   Min negatives: 20
   Max negatives: 20

🔍 EXAMPLE ENTRY:
   Base query: 'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  owl:  <http://www.w3.org/2002/07...'
   Positive queries: 3
     Positive 1: J=1.000, C=0.987
     Query: 'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  owl:  <http:...'
     Positive 2: J=1.000, C=0.984
     Query: 'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  owl:  <http:...'
   Negative queries: 20
     Negative 1: J≈0.050, C≈0.150
     Query: 'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  dbpr: <http:...'
     Negative 2: J≈0.050, C≈0.150
     Query: 'PREFIX  rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX  dbpo: <http://dbpe

### Save the Final Dataset:

In [7]:
import os
# Cell 6: Save Final Dataset
print("💾 SAVING FINAL DATASET")
print("="*60)

# Save as JSON (preserves nested structure)
output_json_path = "results/query_triplet_dataset.json"
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(final_dataset, f, indent=2, ensure_ascii=False)

print(f"✅ Saved JSON dataset to: {output_json_path}")
print(f"   Entries: {len(final_dataset):,}")
print(f"   File size: {os.path.getsize(output_json_path) / (1024*1024):.1f} MB")

# Also save as CSV (flattened for easier viewing)
csv_rows = []
for entry in final_dataset:
    base_query = entry['base_query']
    
    # Add positive pairs
    for pos in entry['positive_queries']:
        csv_rows.append({
            'base_query': base_query,
            'query': pos['query'],
            'type': 'positive',
            'jaccard_score': pos['jaccard_score'],
            'cosine_score': pos['cosine_score']
        })
    
    # Add negative pairs
    for neg in entry['negative_queries']:
        csv_rows.append({
            'base_query': base_query,
            'query': neg['query'],
            'type': 'negative',
            'jaccard_score': neg['assumed_jaccard'],
            'cosine_score': neg['assumed_cosine']
        })

# Create and save CSV
output_csv_path = "results/query_pairs_with_labels.csv"
csv_df = pd.DataFrame(csv_rows)
csv_df.to_csv(output_csv_path, index=False, encoding='utf-8')

print(f"\n✅ Saved flattened CSV to: {output_csv_path}")
print(f"   Total pairs: {len(csv_df):,}")
print(f"   Positive pairs: {len(csv_df[csv_df['type'] == 'positive']):,}")
print(f"   Negative pairs: {len(csv_df[csv_df['type'] == 'negative']):,}")
print(f"   File size: {os.path.getsize(output_csv_path) / (1024*1024):.1f} MB")

# Save statistics
stats = {
    'total_entries': len(final_dataset),
    'total_pairs': len(csv_df),
    'positive_pairs': len(csv_df[csv_df['type'] == 'positive']),
    'negative_pairs': len(csv_df[csv_df['type'] == 'negative']),
    'avg_positives_per_base': avg_positives if final_dataset else 0,
    'avg_negatives_per_base': avg_negatives if final_dataset else 0,
    'positive_thresholds': {
        'jaccard_min': POSITIVE_JACCARD_MIN,
        'cosine_min': POSITIVE_COSINE_MIN
    },
    'negative_thresholds': {
        'jaccard_max': NEGATIVE_JACCARD_MAX,
        'cosine_max': NEGATIVE_COSINE_MAX
    }
}

stats_path = "results/dataset_statistics.json"
with open(stats_path, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2)

print(f"\n✅ Saved statistics to: {stats_path}")
print("\n📊 FINAL STATISTICS:")
for key, value in stats.items():
    if key not in ['positive_thresholds', 'negative_thresholds']:
        if isinstance(value, (int, float)):
            print(f"   {key}: {value:,}" if isinstance(value, int) else f"   {key}: {value:.2f}")

💾 SAVING FINAL DATASET
✅ Saved JSON dataset to: results/query_triplet_dataset.json
   Entries: 4,860
   File size: 55.0 MB

✅ Saved flattened CSV to: results/query_pairs_with_labels.csv
   Total pairs: 145,244
   Positive pairs: 48,044
   Negative pairs: 97,200
   File size: 82.6 MB

✅ Saved statistics to: results/dataset_statistics.json

📊 FINAL STATISTICS:
   total_entries: 4,860
   total_pairs: 145,244
   positive_pairs: 48,044
   negative_pairs: 97,200
   avg_positives_per_base: 9.89
   avg_negatives_per_base: 20.00


### Create Training-Ready Format:

In [8]:
# Cell 7: Create Training-Ready Format (For ML Models)
print("🤖 CREATING TRAINING-READY FORMAT")
print("="*60)

# For training contrastive learning or triplet loss models, we need:
# - anchor (base query)
# - positive (similar query)
# - negative (dissimilar query)

training_triplets = []

for entry in final_dataset:
    base_query = entry['base_query']
    positives = entry['positive_queries']
    negatives = entry['negative_queries']
    
    # Create multiple triplets by pairing each positive with each negative
    # (or sample to avoid combinatorial explosion)
    max_triplets_per_base = 100  # Limit to avoid too many triplets
    
    triplets = []
    for pos in positives:
        for neg in negatives:
            triplets.append({
                'anchor': base_query,
                'positive': pos['query'],
                'negative': neg['query'],
                'positive_jaccard': pos['jaccard_score'],
                'positive_cosine': pos['cosine_score'],
                'negative_jaccard': neg['assumed_jaccard'],
                'negative_cosine': neg['assumed_cosine']
            })
    
    # Sample if too many
    if len(triplets) > max_triplets_per_base:
        triplets = random.sample(triplets, max_triplets_per_base)
    
    training_triplets.extend(triplets)

print(f"✅ Created {len(training_triplets):,} training triplets")

# Save training triplets
triplets_path = "results/training_triplets.json"
with open(triplets_path, 'w', encoding='utf-8') as f:
    json.dump(training_triplets, f, indent=2, ensure_ascii=False)

print(f"✅ Saved training triplets to: {triplets_path}")
print(f"   File size: {os.path.getsize(triplets_path) / (1024*1024):.1f} MB")

# Also save as CSV for easier loading
triplets_csv_path = "results/training_triplets.csv"
triplets_df = pd.DataFrame(training_triplets)
triplets_df.to_csv(triplets_csv_path, index=False, encoding='utf-8')

print(f"✅ Saved training triplets CSV to: {triplets_csv_path}")

# Show example triplets
print(f"\n🔍 EXAMPLE TRIPLETS:")
for i in range(min(3, len(training_triplets))):
    triplet = training_triplets[i]
    print(f"\nTriplet {i+1}:")
    print(f"  Anchor: '{triplet['anchor'][:80]}...'")
    print(f"  Positive (J={triplet['positive_jaccard']:.3f}, C={triplet['positive_cosine']:.3f}):")
    print(f"    '{triplet['positive'][:80]}...'")
    print(f"  Negative (J≈{triplet['negative_jaccard']:.3f}, C≈{triplet['negative_cosine']:.3f}):")
    print(f"    '{triplet['negative'][:80]}...'")

🤖 CREATING TRAINING-READY FORMAT
✅ Created 362,740 training triplets
✅ Saved training triplets to: results/training_triplets.json
   File size: 362.8 MB
✅ Saved training triplets CSV to: results/training_triplets.csv

🔍 EXAMPLE TRIPLETS:

Triplet 1:
  Anchor: 'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  owl:  <http:...'
  Positive (J=1.000, C=0.987):
    'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  owl:  <http:...'
  Negative (J≈0.050, C≈0.150):
    'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  dbpr: <http:...'

Triplet 2:
  Anchor: 'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  owl:  <http:...'
  Positive (J=1.000, C=0.987):
    'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  owl:  <http:...'
  Negative (J≈0.050, C≈0.150):
    'PREFIX  rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX  dbpo: <http://dbpe...'

Triplet 3:
  Anchor: 'PREFIX  rdf:  <http://www.w3.org/1999/0

### Verification and Quality Check:

In [9]:
# Cell 8: Verification and Quality Check
print("✅ VERIFICATION AND QUALITY CHECK")
print("="*60)

# Load the generated dataset to verify
with open(output_json_path, 'r', encoding='utf-8') as f:
    loaded_dataset = json.load(f)

print(f"✅ Loaded dataset with {len(loaded_dataset):,} entries")

# Check quality metrics
quality_issues = []

for entry in loaded_dataset[:100]:  # Check first 100 entries
    # Check positive queries have high similarity
    for pos in entry['positive_queries']:
        if pos['jaccard_score'] < POSITIVE_JACCARD_MIN:
            quality_issues.append(f"Positive Jaccard too low: {pos['jaccard_score']:.3f}")
        if pos['cosine_score'] < POSITIVE_COSINE_MIN:
            quality_issues.append(f"Positive Cosine too low: {pos['cosine_score']:.3f}")
    
    # Check we have reasonable counts
    if entry['positive_count'] < 1:
        quality_issues.append("No positive queries")
    if entry['negative_count'] < 1:
        quality_issues.append("No negative queries")

if quality_issues:
    print(f"⚠️  Found {len(quality_issues)} quality issues in sample")
    for issue in quality_issues[:10]:  # Show first 10
        print(f"   - {issue}")
else:
    print("✅ No quality issues found in sample check")

# Check dataset balance
print(f"\n📈 DATASET BALANCE:")
if loaded_dataset:
    pos_counts = [entry['positive_count'] for entry in loaded_dataset]
    neg_counts = [entry['negative_count'] for entry in loaded_dataset]
    
    print(f"   Positive queries:")
    print(f"     Total: {sum(pos_counts):,}")
    print(f"     Average per base: {np.mean(pos_counts):.1f}")
    print(f"     Std dev: {np.std(pos_counts):.1f}")
    
    print(f"\n   Negative queries:")
    print(f"     Total: {sum(neg_counts):,}")
    print(f"     Average per base: {np.mean(neg_counts):.1f}")
    print(f"     Std dev: {np.std(neg_counts):.1f}")
    
    print(f"\n   Ratio (negatives/positives): {sum(neg_counts)/sum(pos_counts):.2f}")

print(f"\n🎯 DATASET READY FOR USE!")
print(f"\nYou can now use these files:")
print(f"1. {output_json_path} - Full dataset with nested structure")
print(f"2. {output_csv_path} - Flattened pairs with labels")
print(f"3. {triplets_path} - Training triplets for ML models")
print(f"4. {stats_path} - Dataset statistics")

✅ VERIFICATION AND QUALITY CHECK
✅ Loaded dataset with 4,860 entries
✅ No quality issues found in sample check

📈 DATASET BALANCE:
   Positive queries:
     Total: 48,044
     Average per base: 9.9
     Std dev: 7.9

   Negative queries:
     Total: 97,200
     Average per base: 20.0
     Std dev: 0.0

   Ratio (negatives/positives): 2.02

🎯 DATASET READY FOR USE!

You can now use these files:
1. results/query_triplet_dataset.json - Full dataset with nested structure
2. results/query_pairs_with_labels.csv - Flattened pairs with labels
3. results/training_triplets.json - Training triplets for ML models
4. results/dataset_statistics.json - Dataset statistics
